In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TestSQL") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [2]:
customers = spark.read.parquet(
    "hdfs://localhost:9000/output_test/customers"
)

customers.count()

1070

In [4]:
shipments = spark.read.parquet(
    "hdfs://localhost:9000/output_test/shipments"
)

shipments.count()

1051

In [7]:
shipments.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- delivery_date: string (nullable = true)
 |-- shipping_fee: float (nullable = true)



In [10]:
shipments.createOrReplaceTempView("shipments")

In [11]:
spark.sql("""
SELECT avg(shipping_fee) as avg_shipping_fee
FROM shipments
""").show()

+-----------------+
| avg_shipping_fee|
+-----------------+
|4.957608342880072|
+-----------------+



In [5]:
topics = [
    "customers", "products", "promotions", "geography",
    "orders", "order_items", "payments", "shipments",
    "returns", "reviews", "sales", "inventory", "web_traffic"
]

for topic in topics:
    path = f"hdfs://localhost:9000/output_test/{topic}"
    try:
        count = spark.read.parquet(path).count()
        print(topic, count)
    except Exception as e:
        print(topic, "no data")

customers 1070
products 1049
promotions 198
geography 1052
orders 1053
order_items 1051
payments 1052
shipments 1051
returns 1051
reviews 1051
sales 1050
inventory 1050
web_traffic 1051


In [11]:
topics = [
    "customers", "products", "promotions", "geography",
    "orders", "order_items", "payments", "shipments",
    "returns", "reviews", "sales", "inventory", "web_traffic"
]

dfs = {}

for topic in topics:
    path = f"hdfs://localhost:9000/data/{topic}.csv"
    try:
        df = spark.read.csv(
            path,
            header=True,
            inferSchema=True
        )
        dfs[topic] = df
        print(topic, df.count(), len(df.columns))
    except Exception as e:
        print(topic, "no data", e)

customers 121930 7
products 2412 8
promotions 50 10
geography 39948 4
orders 646945 8
order_items 714669 7
payments 646945 4
shipments 566067 4
returns 39939 7
reviews 113551 7
sales 3833 3
inventory 60247 17
web_traffic 3652 7
